In [39]:
import torch
import torch.nn as nn
import torch.optim as optim

In [40]:
# ======================================
# Weather Dataset
# ======================================


X = [
    [30, 32, 31, 33, 35],
    [20, 21, 22, 23, 24],
    [15, 16, 17, 18, 19],
    [40, 41, 42, 43, 44],
    [25, 27, 29, 31, 33]
]

Y = [
    [36, 37, 38],
    [25, 26, 27],
    [20, 21, 22],
    [45, 46, 47],
    [35, 37, 39]
]

X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
Y = torch.tensor(Y, dtype=torch.float32).unsqueeze(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
    

X shape: torch.Size([5, 5, 1])
Y shape: torch.Size([5, 3, 1])


In [41]:
# ======================================
# Encoder
# ======================================

class Encoder(nn.Module):
    def __init__(self,input_size, hidden_size, num_layers=1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self,x):
        outputs, (hidden, cell) = self.lstm(x)
        print("self.lstm(x)",self.lstm(x))
        return hidden,cell
    

In [42]:
# ======================================
# Decoder
# ======================================

class Decoder(nn.Module):
    def __init__(self,input_size, hidden_size, num_layers=1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self,x, hidden, cell):
        output, (hidden, cell) = self.lstm(x, (hidden, cell))
        prediction = self.fc(output)
        return prediction, hidden, cell

In [43]:
# ======================================
# Seq2Seq
# ======================================

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, output_length):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.output_length = output_length

    def forward(self, src):

        batch_size = src.shape[0]

        hidden, cell = self.encoder(src)

        # Start decoder with last value
        decoder_input = src[:, -1:, :]

        outputs = []

        for _ in range(self.output_length):

            prediction, hidden, cell = self.decoder(
                decoder_input,
                hidden,
                cell
            )

            outputs.append(prediction)

            # Feed prediction into next step
            decoder_input = prediction

        outputs = torch.cat(outputs, dim=1)

        return outputs

In [44]:
# ======================================
# Model
# ======================================

INPUT_SIZE = 1
HIDDEN_SIZE = 8
OUTPUT_LENGTH = 3

encoder = Encoder(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE
)

decoder = Decoder(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE
)

model = Seq2Seq(
    encoder,
    decoder,
    OUTPUT_LENGTH
)


In [45]:

# ======================================
# Loss + Optimizer
# ======================================

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)


# ======================================
# Training
# ======================================

epochs = 1

for epoch in range(epochs):

    optimizer.zero_grad()

    predictions = model(X)

    loss = criterion(predictions, Y)

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:4d} | Loss = {loss.item():.6f}"
        )



self.lstm(x) (tensor([[[ 2.3988e-01, -9.7094e-02,  4.5077e-03,  9.6133e-03, -5.9637e-03,
          -9.4418e-05, -2.2766e-06, -2.5648e-01],
         [ 2.3608e-01, -1.7475e-01,  3.0706e-03,  1.6909e-02, -6.1368e-03,
          -4.8316e-05, -3.0360e-06, -3.0993e-01],
         [ 2.4607e-01, -2.4686e-01,  3.7590e-03,  2.5486e-02, -7.6471e-03,
          -6.3756e-05, -5.0702e-06, -3.3332e-01],
         [ 2.3823e-01, -3.0207e-01,  2.5786e-03,  3.2240e-02, -5.5478e-03,
          -3.4491e-05, -4.7078e-06, -3.2304e-01],
         [ 2.2927e-01, -3.4498e-01,  1.7466e-03,  3.7584e-02, -3.9536e-03,
          -1.8799e-05, -3.9847e-06, -3.1082e-01]],

        [[ 2.9701e-01, -1.7284e-01,  3.0598e-02,  2.9381e-02, -3.0258e-02,
          -1.8340e-03, -1.3546e-04, -3.1865e-01],
         [ 3.0521e-01, -3.0011e-01,  2.7176e-02,  5.5065e-02, -3.9570e-02,
          -1.2507e-03, -2.3445e-04, -3.9831e-01],
         [ 3.0616e-01, -3.8833e-01,  2.2789e-02,  7.8578e-02, -3.6047e-02,
          -8.9300e-04, -2.7955e-04

In [46]:
# ======================================
# Prediction
# ======================================

test_sequence = [
    [30, 32, 31, 33, 35]
]

test = torch.tensor(
    test_sequence,
    dtype=torch.float32
).unsqueeze(-1)

model.eval()

with torch.no_grad():

    forecast = model(test)

print("\nInput:")
print(test.squeeze().tolist())

print("\nPredicted Next 3 Days:")
print(forecast.squeeze().numpy())

self.lstm(x) (tensor([[[ 1.9061e-01, -7.2526e-02,  6.1974e-03,  5.6307e-03, -8.2360e-03,
          -6.8526e-05, -4.3120e-06, -2.0517e-01],
         [ 1.8255e-01, -1.2753e-01,  4.3862e-03,  9.6662e-03, -8.4623e-03,
          -3.4760e-05, -5.8245e-06, -2.4218e-01],
         [ 1.9137e-01, -1.7893e-01,  5.2940e-03,  1.4600e-02, -1.0371e-02,
          -4.6601e-05, -9.6401e-06, -2.6213e-01],
         [ 1.8108e-01, -2.1647e-01,  3.7117e-03,  1.8115e-02, -7.6685e-03,
          -2.4812e-05, -9.1638e-06, -2.4835e-01],
         [ 1.7033e-01, -2.4423e-01,  2.5603e-03,  2.0726e-02, -5.5642e-03,
          -1.3320e-05, -7.9233e-06, -2.3369e-01]]]), (tensor([[[ 1.7033e-01, -2.4423e-01,  2.5603e-03,  2.0726e-02, -5.5642e-03,
          -1.3320e-05, -7.9233e-06, -2.3369e-01]]]), tensor([[[ 1.0006e+00, -2.4932e-01,  2.5604e-03,  6.0818e-02, -4.8881e+00,
          -1.3329e-05, -4.1298e-04, -4.9998e+00]]])))

Input:
[30.0, 32.0, 31.0, 33.0, 35.0]

Predicted Next 3 Days:
[-0.19138464 -0.01919574 -0.14752346]